# 40 — Generate `usdm_v4.context.jsonld`

A JSON-LD 1.1 context that makes USDM API instance JSON (the wire format
exchanged by DDF-compliant systems) readable as RDF against the
`usdm_v4.ttl` ontology. Generated mechanically from
`../downloads/dataStructure.yml` — same source, same pinned DDF-RA tag as
the Turtle deliverable. Decision D5 in `docs/iri-and-governance.md`.

**Mapping summary**

| Instance JSON                          | JSON-LD                                                        |
|----------------------------------------|----------------------------------------------------------------|
| `id`                                   | `@id` (no `@base` in the context — ids expand relative to the consuming document) |
| `instanceType`                         | `@type`; class names are context terms mapping to `usdm:{ClassName}` |
| serialization key on a typed object    | type-scoped context term → `usdm:{DeclaringClass}-{attributeName}` |
| `Relationship Type: Ref` attribute     | `"@type": "@id"` — id-string cross-references become links   |
| `date` / `float` primitive             | `"@type": "xsd:date"` / `"xsd:float"` (other primitives map natively) |
| max cardinality `*`                    | `"@container": "@set"`                                       |

Consequence of the aliasing: the `{Class}-id` and `{Class}-instanceType`
property IRIs never appear in instance graphs derived via this context —
node identity and `rdf:type` carry that information instead.


## 1. Load source YAML


In [1]:
import yaml
from pathlib import Path

YAML_PATH = "../downloads/dataStructure.yml"
if not Path(YAML_PATH).exists():
    raise FileNotFoundError(
        f"{YAML_PATH} missing — run notebooks/10_fetch_yaml.ipynb first."
    )

with open(YAML_PATH) as f:
    MODEL = yaml.safe_load(f)

print(f"{len(MODEL)} top-level entries")


86 top-level entries


## 2. Declaring-class resolution

The YAML re-lists inherited attributes on subclasses, marked with
`Inherited From`. `20_generate_turtle.ipynb` emits a property IRI only on
the declaring class and skips marked attributes. The context must point
every serialization key at that same declaring-class IRI, so the
resolution here follows the `Inherited From` chain until it reaches an
unmarked declaration.

Counterexample that proves the rule: `analysisPopulations` appears on
`StudyDesign` *and* both of its subclasses **without** the marker, so the
ontology declares all three property IRIs — and each subclass context
maps the key to its own IRI, not the parent's.


In [2]:
def resolve_ref(ref):
    return ref.lstrip("#/")


def super_classes(class_name):
    entry = MODEL[class_name]
    return [resolve_ref(r["$ref"]) for r in (entry.get("Super Classes") or [])]


def declaring_class(class_name, key):
    attrs = MODEL[class_name].get("Attributes") or {}
    if key in attrs:
        inherited_from = attrs[key].get("Inherited From")
        if not inherited_from:
            return class_name
        if len(inherited_from) != 1:
            raise ValueError(
                f"{class_name}.{key}: multi-parent Inherited From {inherited_from}"
            )
        return declaring_class(resolve_ref(inherited_from[0]["$ref"]), key)
    hits = set()
    for sc in super_classes(class_name):
        try:
            hits.add(declaring_class(sc, key))
        except KeyError:
            pass
    if len(hits) != 1:
        raise KeyError(
            f"{class_name}.{key}: declaring class not unique: {sorted(hits)}"
        )
    return hits.pop()


def all_keys(class_name):
    keys = set(MODEL[class_name].get("Attributes") or {})
    for sc in super_classes(class_name):
        keys |= all_keys(sc)
    return keys


## 3. Term definitions

One term per serialization key per class. `id` and `instanceType` are
aliased at the root and skipped here.


In [3]:
PRIMITIVES = {"string", "boolean", "date", "float", "integer"}


def term_definition(class_name, key):
    declaring = declaring_class(class_name, key)
    attr = (MODEL[declaring].get("Attributes") or {})[key]
    term = {"@id": f"usdm:{declaring}-{key}"}
    refs = [resolve_ref(t["$ref"]) for t in attr["Type"]]
    if attr.get("Relationship Type") == "Ref":
        term["@type"] = "@id"
    elif len(refs) == 1 and refs[0] == "date":
        term["@type"] = "xsd:date"
    elif len(refs) == 1 and refs[0] == "float":
        term["@type"] = "xsd:float"
    if (attr.get("Cardinality") or "").endswith("*"):
        term["@container"] = "@set"
    return term


## 4. Assemble context and write `../usdm_v4.context.jsonld`

Baseline counts (DDF-RA v4.0.0) are asserted fail-fast, mirroring the
baseline style of `30_validate.ipynb`.


In [4]:
import json

USDM = "https://w3id.org/cdisc/usdm/v4/"
XSD = "http://www.w3.org/2001/XMLSchema#"
OUT_PATH = "../usdm_v4.context.jsonld"

root = {
    "@version": 1.1,
    "usdm": USDM,
    "xsd": XSD,
    "id": "@id",
    "instanceType": "@type",
}

stats = {"classes": 0, "mapped_keys": 0, "ref_coercions": 0,
         "date_coercions": 0, "float_coercions": 0, "set_containers": 0}

for class_name in sorted(MODEL):
    scoped = {}
    for key in sorted(all_keys(class_name)):
        if key in ("id", "instanceType"):
            continue
        term = term_definition(class_name, key)
        scoped[key] = term
        stats["mapped_keys"] += 1
        coercion = term.get("@type")
        if coercion == "@id":
            stats["ref_coercions"] += 1
        elif coercion == "xsd:date":
            stats["date_coercions"] += 1
        elif coercion == "xsd:float":
            stats["float_coercions"] += 1
        if "@container" in term:
            stats["set_containers"] += 1
    root[class_name] = {"@id": f"usdm:{class_name}", "@context": scoped}
    stats["classes"] += 1

BASELINE = {
    "classes": 86,
    "mapped_keys": 667,
    "ref_coercions": 83,
    "date_coercions": 1,
    "float_coercions": 1,
    "set_containers": 262,
}
for k, expected in BASELINE.items():
    assert stats[k] == expected, f"{k}: expected {expected}, got {stats[k]}"

with open(OUT_PATH, "w") as f:
    json.dump({"@context": root}, f, indent=1, ensure_ascii=False)
    f.write("\n")

print(json.dumps(stats, indent=1))
print(f"wrote {OUT_PATH}")


{
 "classes": 86,
 "mapped_keys": 667,
 "ref_coercions": 83,
 "date_coercions": 1,
 "float_coercions": 1,
 "set_containers": 262
}
wrote ../usdm_v4.context.jsonld


## 5. Cross-check against `usdm_v4.ttl`

Every property IRI the context references must be declared in the
ontology, and every declared property the context does *not* reference
must be a `{Class}-id` or `{Class}-instanceType` property (absorbed by
the `@id` / `@type` aliases).


In [5]:
import rdflib
from rdflib.namespace import OWL, RDF

TTL_PATH = "../usdm_v4.ttl"
if not Path(TTL_PATH).exists():
    raise FileNotFoundError(
        f"{TTL_PATH} missing — run notebooks/20_generate_turtle.ipynb first."
    )

ont = rdflib.Graph()
ont.parse(TTL_PATH, format="turtle")
declared = {
    str(s)[len(USDM):]
    for prop_type in (OWL.DatatypeProperty, OWL.ObjectProperty)
    for s in ont.subjects(RDF.type, prop_type)
}

referenced = set()
for class_def in root.values():
    if isinstance(class_def, dict) and "@context" in class_def:
        for term in class_def["@context"].values():
            referenced.add(term["@id"][len("usdm:"):])

missing = referenced - declared
assert not missing, f"context references undeclared properties: {sorted(missing)[:10]}"

unreferenced = declared - referenced
bad = {p for p in unreferenced if p.rsplit("-", 1)[1] not in ("id", "instanceType")}
assert not bad, f"unreferenced non-alias properties: {sorted(bad)[:10]}"

print(f"declared properties:    {len(declared)}")
print(f"referenced by context:  {len(referenced)}")
print(f"unreferenced (id/instanceType, absorbed by aliases): {len(unreferenced)}")


declared properties:    693
referenced by context:  545
unreferenced (id/instanceType, absorbed by aliases): 148


## Provenance

Generated by `notebooks/40_generate_context.ipynb` in
[kerfors/usdm-rdf](https://github.com/kerfors/usdm-rdf) from
`downloads/dataStructure.yml` (DDF-RA tag pinned in
`notebooks/10_fetch_yaml.ipynb`).
